# Data Preparation
Ce notebook combine les CSV produits par `data_scrapping.ipynb` en un unique dataset prêt à l'emploi.

**Fichiers requis (produits par scrapping) :**
- `data/games.csv`
- `data/managers.csv`
- `data/managers_manquants.csv` — managers absents de l'API, attributs tactiques uniquement
- `data/managers_manquants2.csv` — managers API avec stats NaN, attributs tactiques + stats partielles
- `data/meteo.csv`
- `data/referees.csv`
- `data/transfermarkt_managers_history.csv`

**Fichier produit :**
- `data/dataset_final.csv`

**Ordre d'exécution :**
1. Setup & chargement
2. Merge Météo
3. Merge Arbitres
4. Attribution des Managers (Transfermarkt)
5. Construction du référentiel Managers complet
6. Merge Managers
7. Export final

## 1. Setup & Chargement

In [60]:
import os
import pandas as pd
from datetime import datetime

# ── Chargement des fichiers sources ───────────────────────────────────────────
games    = pd.read_csv("data/games.csv")
meteo    = pd.read_csv("data/meteo.csv")
managers = pd.read_csv("data/managers.csv")
managers_manquants  = pd.read_csv("data/managers_manquants.csv")   # managers absents de l'API
managers_manquants2 = pd.read_csv("data/managers_manquants2.csv")  # managers API avec stats NaN
referees = pd.read_csv("data/referees.csv")
tm_hist  = pd.read_csv("data/transfermarkt_managers_history.csv")

print("📂 Fichiers chargés :")
print(f"  games               : {games.shape}")
print(f"  meteo               : {meteo.shape}")
print(f"  managers            : {managers.shape}")
print(f"  managers_manquants  : {managers_manquants.shape}")
print(f"  managers_manquants2 : {managers_manquants2.shape}")
print(f"  referees            : {referees.shape}")
print(f"  tm_hist             : {tm_hist.shape}")

📂 Fichiers chargés :
  games               : (8721, 10)
  meteo               : (10225, 9)
  managers            : (1439, 36)
  managers_manquants  : (214, 6)
  managers_manquants2 : (44, 9)
  referees            : (1393, 5)
  tm_hist             : (8934, 5)


## 2. Merge Météo

In [61]:
# Suppression des matchs sans score (non joués)
games = games.dropna(subset=["Score"])

# Merge sur (Home, Away, Date)
df = games.merge(
    meteo,
    left_on=["Home", "Away", "Date"],
    right_on=["Home_Team", "Away_Team", "Date"],
    how="left"
).drop(columns=["Home_Team", "Away_Team"])

print(f"✅ Après merge météo : {df.shape}")
print(f"   Matchs sans météo : {df['Temp_Moy_C'].isna().sum()}")

✅ Après merge météo : (8722, 16)
   Matchs sans météo : 0


## 3. Merge Arbitres

In [62]:
# Colonnes utiles des arbitres
COLS_REF = ["name", "avg_yellow_per_match", "avg_red_per_match",
            "avg_goals_per_match", "avg_fouls_per_match"]
referees_clean = referees[[c for c in COLS_REF if c in referees.columns]].copy()

# Vérification couverture
manquants_ref = df[~df["Referee"].isin(referees_clean["name"])]["Referee"].dropna().unique()
if len(manquants_ref) > 0:
    print(f"⚠️ {len(manquants_ref)} arbitres sans stats (seront en NaN) : {manquants_ref[:10]}")
else:
    print("✅ Tous les arbitres sont couverts")

# Merge
df = df.merge(referees_clean, left_on="Referee", right_on="name", how="left").drop(columns=["name"])

print(f"✅ Après merge arbitres : {df.shape}")

✅ Tous les arbitres sont couverts
✅ Après merge arbitres : (8722, 20)


In [63]:
referees_clean.to_csv("data/referees_processed.csv")

## 4. Attribution des Managers par date

In [64]:
TODAY_STR = datetime.now().strftime('%Y-%m-%d')
MOIS_FR = {
    'janv.': 'Jan', 'févr.': 'Feb', 'mars': 'Mar', 'avr.': 'Apr',
    'mai': 'May', 'juin': 'Jun', 'juil.': 'Jul', 'août': 'Aug',
    'sept.': 'Sep', 'oct.': 'Oct', 'nov.': 'Nov', 'déc.': 'Dec'
}

def clean_tm_date(date_str, is_departure=False):
    """Convertit une date Transfermarkt (français) en YYYY-MM-DD."""
    if pd.isna(date_str) or str(date_str).strip() in ["", "-", "nan"]:
        return TODAY_STR if is_departure else None
    new_date = str(date_str).lower()
    for fr, en in MOIS_FR.items():
        if fr in new_date:
            new_date = new_date.replace(fr, en)
            break
    try:
        return pd.to_datetime(new_date, dayfirst=True).strftime('%Y-%m-%d')
    except Exception:
        return TODAY_STR if is_departure else None


# Nettoyage de l'historique Transfermarkt
tm = tm_hist.copy()
tm['Nommé']  = tm['Nommé'].apply(lambda x: clean_tm_date(x, is_departure=False))
tm['Départ'] = tm['Départ'].apply(lambda x: clean_tm_date(x, is_departure=True))
tm['Nommé']  = pd.to_datetime(tm['Nommé'],  errors='coerce')
tm['Départ'] = pd.to_datetime(tm['Départ'], errors='coerce')

df['Date'] = pd.to_datetime(df['Date'])


def get_manager(team_name, match_date, tm_db):
    """Retourne le manager en poste à la date du match (fallback au plus proche)."""
    team_managers = tm_db[tm_db['Club'] == team_name].copy()
    if team_managers.empty:
        return "Unknown"

    exact = team_managers[
        (team_managers['Nommé'] <= match_date) &
        (team_managers['Départ'] >= match_date)
    ]
    if not exact.empty:
        return exact.iloc[0]['Manager']

    # Fallback : manager avec la date de mandat la plus proche
    team_managers['dist'] = team_managers[['Nommé', 'Départ']].apply(
        lambda r: min(
            abs((r['Nommé'] - match_date).days) if pd.notna(r['Nommé']) else float('inf'),
            abs((r['Départ'] - match_date).days) if pd.notna(r['Départ']) else float('inf')
        ), axis=1
    )
    return team_managers.sort_values('dist').iloc[0]['Manager']


print("🔄 Attribution Home Manager...")
df['Home_Manager'] = df.apply(lambda r: get_manager(r['Home'], r['Date'], tm), axis=1)

print("🔄 Attribution Away Manager...")
df['Away_Manager'] = df.apply(lambda r: get_manager(r['Away'], r['Date'], tm), axis=1)

rate_home = (df['Home_Manager'] != 'Unknown').mean() * 100
rate_away = (df['Away_Manager'] != 'Unknown').mean() * 100
print(f"✅ Taux de succès — Home: {rate_home:.1f}% | Away: {rate_away:.1f}%")

C:\Users\MUEL\AppData\Local\Temp\ipykernel_14604\1760763264.py:18: UserWarning: Parsing dates in %Y-%m-%d format when dayfirst=True was specified. Pass `dayfirst=False` or specify a format to silence this warning.
  return pd.to_datetime(new_date, dayfirst=True).strftime('%Y-%m-%d')
C:\Users\MUEL\AppData\Local\Temp\ipykernel_14604\1760763264.py:18: UserWarning: Parsing dates in %Y-%m-%d format when dayfirst=True was specified. Pass `dayfirst=False` or specify a format to silence this warning.
  return pd.to_datetime(new_date, dayfirst=True).strftime('%Y-%m-%d')


🔄 Attribution Home Manager...
🔄 Attribution Away Manager...
✅ Taux de succès — Home: 100.0% | Away: 100.0%


## 5. Construction du référentiel Managers complet

In [65]:
# ── Colonnes à conserver depuis l'API (les autres sont trop spécifiques ou redondantes) ────
COLS_TO_DROP_MGR = [
    'id', 'short_name', 'country', 'formations_used', 'tactical_styles',
    'matches_total', 'btts_pct', 'over_25_pct', 'over_15_pct',
    'avg_goals_scored_1h', 'avg_goals_conceded_1h', 'current_team_id',
    'wins', 'draws', 'losses', 'avg_goals_scored', 'avg_goals_conceded',
    'avg_possession', 'avg_shots', 'avg_shots_on_target', 'avg_xg_for',
    'avg_xg_against', 'avg_corners', 'avg_yellow_cards', 'avg_red_cards',
    'avg_fouls', 'current_team_name',
]
# Colonnes finales : name, preferred_formation, profile, team_style,
#                   pressing_intensity, defensive_line,
#                   win_pct, clean_sheet_pct, fail_to_score_pct
COLS_FINAL = ['name', 'preferred_formation', 'profile', 'team_style',
              'pressing_intensity', 'defensive_line',
              'win_pct', 'clean_sheet_pct', 'fail_to_score_pct']

managers_clean = managers.drop(
    columns=[c for c in COLS_TO_DROP_MGR if c in managers.columns]
).copy()

print(f"managers API après nettoyage  : {managers_clean.shape[0]} entrées")

managers API après nettoyage  : 1439 entrées


In [66]:
# ── Fonction de calcul des stats depuis les scores des matchs ───────────────
def calc_manager_stats(name, df_games):
    """Calcule win_pct, clean_sheet_pct, fail_to_score_pct à partir des scores."""
    home_games = df_games[df_games['Home_Manager'] == name]
    away_games = df_games[df_games['Away_Manager'] == name]
    total = len(home_games) + len(away_games)

    if total == 0:
        return None, None, None

    wins, clean_sheets, fail_to_score = 0, 0, 0

    for _, row in home_games.iterrows():
        score = str(row['Score']).replace('–', '-').replace('—', '-')
        parts = score.split('-')
        if len(parts) != 2:
            continue
        try:
            h, a = int(parts[0].strip()), int(parts[1].strip())
        except ValueError:
            continue
        if h > a: wins += 1
        if a == 0: clean_sheets += 1
        if h == 0: fail_to_score += 1

    for _, row in away_games.iterrows():
        score = str(row['Score']).replace('–', '-').replace('—', '-')
        parts = score.split('-')
        if len(parts) != 2:
            continue
        try:
            h, a = int(parts[0].strip()), int(parts[1].strip())
        except ValueError:
            continue
        if a > h: wins += 1
        if h == 0: clean_sheets += 1
        if a == 0: fail_to_score += 1

    return (
        round(wins / total * 100, 2),
        round(clean_sheets / total * 100, 2),
        round(fail_to_score / total * 100, 2),
    )

print("✅ Fonction calc_manager_stats définie")

✅ Fonction calc_manager_stats définie


In [67]:
# ── managers_manquants : absents de l'API, stats à calculer ─────────────────
# Renommage pour harmoniser avec les autres fichiers
mm1 = managers_manquants.rename(columns={'Manager_Name': 'name'}).copy()

print(f"🔄 Calcul des stats pour {len(mm1)} managers manquants...")
for idx, row in mm1.iterrows():
    win, cs, fts = calc_manager_stats(row['name'], df)
    mm1.loc[idx, 'win_pct']          = win
    mm1.loc[idx, 'clean_sheet_pct']  = cs
    mm1.loc[idx, 'fail_to_score_pct'] = fts

# Alignement sur les colonnes finales
mm1 = mm1.reindex(columns=COLS_FINAL)
print(f"✅ managers_manquants prêt : {mm1.shape}")

🔄 Calcul des stats pour 214 managers manquants...
✅ managers_manquants prêt : (214, 9)


In [68]:
# ── managers_manquants2 : dans l'API mais stats NaN, recalcul ───────────────
mm2 = managers_manquants2.copy()

# On recalcule uniquement les lignes avec au moins une stat NaN
mask_nan = mm2[['win_pct', 'clean_sheet_pct', 'fail_to_score_pct']].isna().any(axis=1)
print(f"🔄 Recalcul des stats pour {mask_nan.sum()} managers (NaN détectés)...")

for idx, row in mm2[mask_nan].iterrows():
    win, cs, fts = calc_manager_stats(row['name'], df)
    mm2.loc[idx, 'win_pct']           = win
    mm2.loc[idx, 'clean_sheet_pct']   = cs
    mm2.loc[idx, 'fail_to_score_pct'] = fts

# Alignement sur les colonnes finales
mm2 = mm2.reindex(columns=COLS_FINAL)
print(f"✅ managers_manquants2 prêt : {mm2.shape}")

🔄 Recalcul des stats pour 22 managers (NaN détectés)...
✅ managers_manquants2 prêt : (44, 9)


In [69]:
# ── Concat des 3 sources en un référentiel unique ───────────────────────────
# managers_manquants2 prend la priorité sur managers_clean pour les noms en commun
# (on supprime leurs doublons de managers_clean avant de concat)
noms_mm2 = set(mm2['name'].dropna())
managers_clean_filtered = managers_clean[~managers_clean['name'].isin(noms_mm2)]

managers_ref = pd.concat(
    [managers_clean_filtered, mm1, mm2],
    ignore_index=True
).drop_duplicates(subset='name', keep='first')

print(f"✅ Référentiel managers complet : {managers_ref.shape[0]} managers uniques")

# Vérification couverture
all_managers_in_games = set(df['Home_Manager'].dropna()) | set(df['Away_Manager'].dropna())
all_managers_in_games.discard('Unknown')
manquants_mgr = all_managers_in_games - set(managers_ref['name'].dropna())
if manquants_mgr:
    print(f"⚠️ {len(manquants_mgr)} managers toujours sans stats : {sorted(manquants_mgr)}")
else:
    print("✅ Tous les managers sont couverts")

✅ Référentiel managers complet : 1636 managers uniques
✅ Tous les managers sont couverts


In [70]:
managers_ref.to_csv("data/processed_managers.csv")

## 6. Merge Managers

In [71]:
# Merge Home Manager
df = df.merge(
    managers_ref,
    left_on='Home_Manager', right_on='name', how='left'
).rename(columns={col: f"{col}_home" for col in managers_ref.columns if col != 'name'}
).drop(columns=['name'], errors='ignore')

# Merge Away Manager
df = df.merge(
    managers_ref,
    left_on='Away_Manager', right_on='name', how='left'
).rename(columns={col: f"{col}_away" for col in managers_ref.columns if col != 'name'}
).drop(columns=['name'], errors='ignore')

print(f"✅ Après merge managers : {df.shape}")

✅ Après merge managers : (8722, 38)


## 7. Export final

In [77]:
print(f"   {df.shape[0]} lignes × {df.shape[1]} colonnes")
print(f"\n📊 Valeurs manquantes par colonne :")
missing = df.isna().sum()
missing = missing[missing > 0].sort_values(ascending=False)
print(missing.to_string() if len(missing) > 0 else "  Aucune")

   8722 lignes × 38 colonnes

📊 Valeurs manquantes par colonne :
Attendance    41


In [78]:
new_df = df.dropna()

In [79]:
new_df.shape

(8681, 38)

In [82]:
output_path = "data/dataset_final.csv"
new_df.to_csv(output_path, index=False)

print(f"🎉 Dataset final sauvegardé : {output_path}")

🎉 Dataset final sauvegardé : data/dataset_final.csv


In [81]:
new_df.head()

,League,Season,Date,Time,Home,Score,Away,Attendance,Venue,Referee,...,clean_sheet_pct_home,fail_to_score_pct_home,preferred_formation_away,profile_away,team_style_away,pressing_intensity_away,defensive_line_away,win_pct_away,clean_sheet_pct_away,fail_to_score_pct_away
0,La Liga,2025-2026,2025-08-15,19:00,Girona,1–3,Rayo Vallecano,12403.0,Estadi Municipal de Montilivi,Javier Alberola Rojas,...,20.41,23.13,4-2-3-1,balanced,hard,0.726109,mid,29.70,27.00,35.10
1,La Liga,2025-2026,2025-08-15,21:30,Villarreal,2–0,Oviedo,18333.0,Estadio de la Cerámica,Alejandro Muñiz Ruiz,...,22.70,25.00,4-4-2,attacking,possession,0.750000,high,33.33,33.33,66.67
2,La Liga,2025-2026,2025-08-16,19:30,Mallorca,0–3,Barcelona,23318.0,Estadi Mallorca Son Moix,José Luis Munuera Montero,...,26.63,34.32,4-2-3-1,attacking,possession,0.860000,high,77.78,41.11,3.33
3,La Liga,2025-2026,2025-08-16,21:30,Alavés,2–1,Levante,12837.0,Estadio de Mendizorroza,Miguel Sesma Espinosa,...,0.00,0.00,4-2-3-1,attacking,gegenpress,0.870000,high,14.29,14.29,28.57
4,La Liga,2025-2026,2025-08-16,21:30,Valencia,1–1,Real Sociedad,45333.0,Estadio de Mestalla,Jose Maria Sanchez Santos,...,28.90,21.10,3-4-3,defensive,counter,0.320000,low,27.03,10.81,37.84
